# Price Prediction LLM Fine-Tuning
This notebook fine-tunes a language model using QLoRA to predict prices.

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

## 1. Load Dataset

In [ ]:
dataset_name = "ed-donner/items_prompts_lite"
dataset = load_dataset(dataset_name)

train_dataset = dataset["train"]
if "text" not in train_dataset.column_names:
    def format_prompt(example):
        instruction = example.get("instruction", example.get("prompt", ""))
        response = example.get("output", example.get("completion", ""))
        return {"text": f"### Instruction:\n{instruction}\n\n### Response:\n{response}"}
    train_dataset = train_dataset.map(format_prompt)

print(train_dataset[0])

## 2. Load Base Model and Apply Quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Using Qwen as a reliable, open model
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

## 3. Configure LoRA (Low Rank Adaptation)

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 4. Setup Training Arguments & SFTTrainer

In [ ]:
training_args = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_steps=100,
    dataset_text_field="text",
    max_length=256,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=training_args,
)

## 5. Train and Save Model

In [ ]:
trainer.train()

trainer.save_model("./finetuned_price_model")
tokenizer.save_pretrained("./finetuned_price_model")
print("Training Complete!")

## 6. Inference (Testing the Fine-tuned Model)

In [ ]:
from peft import PeftModel

fine_tuned_model = PeftModel.from_pretrained(model, "./finetuned_price_model")

instruction = "Estimate the price of a vintage leather jacket from the 1980s in good condition."
prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)
outputs = fine_tuned_model.generate(**inputs, max_new_tokens=10)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))